In [35]:
"""
Dependancy Imports
"""
import yaml
import numpy as np
import sys
import os

# Adds the parent directory of 'src' to the system path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))


def load_config(path="../configs/default.yaml"):
  with open(path, "r") as f:
    return yaml.safe_load(f)
cfg = load_config()

### Implementing the path loss and steering vector
Eq 12. Get the path loss
$$
\beta(d) = \beta_0 \left( \frac{d}{d_0} \right)^{-\eta} 
$$

Eq 11. Get the ULA steering vector

$$
a_Q(\theta) = \frac{1}{\sqrt{Q}} \left[ 1, e^{j \pi \sin \theta}, \dots, e^{j \pi (Q-1) \sin \theta} \right]^T
$$

In [36]:
from src.utils.channel_utils import to_db, to_linear

def lamba():
  return float(cfg["channel_model"]["c"]) / float(cfg["channel_model"]["carrier_frequency"])

def steering_vector(elements: int, angle_rad: float) -> np.ndarray:
  q = np.arange(elements)
  return (1.0 / np.sqrt(elements)) * np.exp(1j * np.pi * q * np.sin(angle_rad))


def get_path_loss_linear(d: float, eta:float):
  return to_linear(int(cfg["channel_model"]["beta_0_dB"])) * ((d / int(cfg["channel_model"]["d_0"])) ** -eta)

### Implement Channel Models
- Rician Fading
- Rayleigh Fading

Equation 10 and 11 implemented

In [40]:
def get_H_bar(rx_elements: int, tx_elements: int, angle_rx_rad: float, angle_tx_rad: float = None) -> np.ndarray:
  a_rx = steering_vector(rx_elements, angle_rx_rad)
  if tx_elements == 1:
    return a_rx.reshape(-1,1)
  
  a_tx = steering_vector(tx_elements, angle_tx_rad)

  return np.outer(a_rx, a_tx.conj())

def get_H_tilde(rx_elements: int, tx_elements: int, rng: np.random.Generator) -> np.ndarray:
  real = rng.standard_normal((rx_elements, tx_elements))
  imag = rng.standard_normal((rx_elements, tx_elements))
  return (real + 1j * imag) / np.sqrt(2.0)

def get_hybird_channel_model(rx_elements:int, tx_elements:int, rician_factor: int, beta:float, angle_rx_rad: float, angle_tx_rad: float = None, rng: np.random.Generator = None) -> np.ndarray:
  h_bar = get_H_bar(rx_elements=rx_elements, tx_elements=tx_elements, 
  angle_rx_rad=angle_rx_rad, angle_tx_rad=angle_tx_rad)
  h_tilde = get_H_tilde(rx_elements=rx_elements, tx_elements=tx_elements, rng=rng)

  temp_term = (beta * rician_factor) / (rician_factor + 1)

  scale_h_bar_to_path_loss_power = np.sqrt(temp_term) * h_bar

  scale_h_tilde_to_path_loss_pwoer = (np.sqrt(beta / (rician_factor + 1))) * h_tilde

  return scale_h_bar_to_path_loss_power + scale_h_tilde_to_path_loss_pwoer


In [43]:
from src.utils.channel_utils import to_db, to_linear
rng = np.random.default_rng(0)
rx = 32
tx = 8
beta = get_path_loss_linear(d = 50.0, eta = cfg["channel_model"]["eta_los"])
rician_factor = to_linear(5.0)

channel = get_hybird_channel_model(rx,tx,rician_factor,beta,0.3, 0.2, rng)
print("channel shape:", channel.shape, "| avg power:", to_db(np.mean(np.abs(channel) ** 2)))
 

channel shape: (32, 8) | avg power: -109.52468939053661


### Implement recieved signal and comm user
Equation 5, 6, 7

In [50]:
def get_Gi(beta_g, kappa_g, g_rx_elements, g_tx_elements, angle_rx, angle_tx, rng):
    """BS<->panel, Eq.(13). g_rx_elements=L (panel), g_tx_elements=M (BS)."""
    return get_hybird_channel_model(g_rx_elements, g_tx_elements, kappa_g, beta_g,
                                     angle_rx, angle_tx, rng)
 
 
def get_fi(beta_f, kappa_f, f_rx_elements, angle_rx, angle_tx, rng):
    """panel<->user, Eq.(14). f_rx_elements=L (panel); user side is always 1."""
    return get_hybird_channel_model(f_rx_elements, 1, kappa_f, beta_f,
                                     angle_rx, angle_tx, rng)
 
 
def get_bi(beta_b, kappa_b, b_rx_elements, angle_rx, angle_tx, rng):
    """panel<->target, Eq.(15). b_rx_elements=L (panel); target side is always 1."""
    return get_hybird_channel_model(b_rx_elements, 1, kappa_b, beta_b,
                                     angle_rx, angle_tx, rng)
 
 
def get_hdk(beta_hdk, rx_elements, angle_rx, rng):
    """Direct BS<->user, Eq.(16). rx_elements=M (BS); kappa=0 exactly (pure NLoS)."""
    return get_hybird_channel_model(rx_elements, 1, 0.0, beta_hdk,
                                     angle_rx, 0.0, rng)

array([[ 2.99792724e-05-3.35604800e-04j],
       [-2.66468833e-04+1.64401413e-04j],
       [ 5.14752968e-04-3.98247913e-04j],
       [-4.82761627e-04+2.49752248e-04j],
       [-5.82013356e-05-5.89034857e-04j],
       [ 1.34086215e-04-2.14483312e-04j],
       [-4.03654696e-05+1.40311227e-05j],
       [-3.10058614e-04-4.83916893e-04j]])

In [ ]:
 
def build_hbar(h_dk, G_list, phi_list, f_list, ris_info_list):
    h_bar_k = h_dk.copy()
    for G_i, phi_i, f_i, ris in zip(G_list, phi_list, f_list, ris_info_list):
        if ris["a"] == 0:
            continue
        h_bar_k = h_bar_k + ris["a"] * (G_i.conj().T @ phi_i.conj().T @ f_i)
    return h_bar_k

def get_received_signal(h_bar_k, w_list, s_list, ris_info_list, phi_list, f_list, sigma_k, rng):
    signal = 0.0 + 0.0j
    for w_j, s_j in zip(w_list, s_list):
        signal += (h_bar_k.conj().T @ w_j).item() * s_j
 
    amplified_noise = 0.0 + 0.0j
    for ris, phi_i, f_i in zip(ris_info_list, phi_list, f_list):
        if not ris["is_active"] or ris["a"] == 0:
            continue
        v_i = ris["v"] if ris["v"] is not None else np.zeros((phi_i.shape[0], 1), dtype=complex)
        amplified_noise += ris["a"] * (f_i.conj().T @ phi_i @ v_i).item()
 
    n_k = ((rng.standard_normal() + 1j * rng.standard_normal()) / np.sqrt(2)) * sigma_k
    return signal + amplified_noise + n_k
